# Stage 9 — Final SVM A1 / B và Test một lần

Dataset: CICFLOWMETER_FEATURE_CSV. A1 step5; B step2/coarse6/fine ±5. EXPERIMENTAL_NON_OFFICIAL.
Selection cũ chỉ đọc. Final scaler mới fit Train+Validation theo source gốc. Không MI/RFE/k-search.
Mặc định preflight, final/Test tắt. Làm từng METHOD; Test có xác nhận riêng, không tự retry claim.
Chạy tuần tự từng cell và đọc trạng thái; không chọn Run all cho một phiên evaluation.


## Cell 1 — Imports và cấu hình


In [ ]:
# Cell 1 — cấu hình, không training.
if globals().get('JOB9', {}).get('running') or globals().get('JOB', {}).get('running'):
    raise RuntimeError('Worker còn chạy; không reset cấu hình/source.')
from pathlib import Path
import json, re, subprocess, sys
sys.dont_write_bytecode = True
DRIVE = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE / 'CICFLOWMETER_FEATURE_CSV'
OUTPUT_ROOT = DRIVE / 'experiment_results'
LOCAL_ROOT = Path('/content/stage9_svm_final')
APPROVED_COMMIT = ''  # SHA40 mới chứa Stage9, sau khi bạn push.
METHOD = 'A1'  # đổi 'B' khi làm phương án B, chạy lại setup trong session không có worker.
CONFIGS = ['FW256_FS8', 'FW256_FS16', 'FW512_FS8', 'FW512_FS16']
SELECTED_CONFIGS = ['FW256_FS8']  # hoặc list(CONFIGS) để chạy tuần tự bốn config của MỘT phương án.
SELECTION_RUN_IDS = {
    'A1': {'FW256_FS8': 'RUN_A1_01_FIX01', 'FW256_FS16': 'RUN_A1_02',
           'FW512_FS8': 'RUN_A1_01', 'FW512_FS16': 'RUN_A1_01'},
    'B': {config: 'RUN_B2_01' for config in CONFIGS},  # đối chiếu Cell 6, sửa đúng tên thực tế.
}
FINAL_RUN_IDS = {method: {config: 'RUN_S9_01' for config in CONFIGS} for method in ('A1', 'B')}
EVIDENCE_ROOTS = {}  # (method, config) -> original small frozen metadata, nếu cần.
PREVIOUS_WORKER_STOPPED = {}  # chỉ điền khi đã xác nhận worker cũ thực sự dừng.
MODE = 'preflight'
ENABLE_FINAL_FIT = False
ENABLE_TEST_EVALUATION = False
TEST_CONFIRMATION = ''
BUDGET = {'min_available_ram_bytes': 2*1024**3, 'max_process_rss_bytes': 40*1024**3}
JOB9 = {'running': False, 'error': None}
if METHOD not in ('A1', 'B') or not SELECTED_CONFIGS or len(set(SELECTED_CONFIGS)) != len(SELECTED_CONFIGS) or not set(SELECTED_CONFIGS) <= set(CONFIGS):
    raise ValueError('METHOD/config không hợp lệ.')
print('METHOD:', METHOD, '| CONFIGS:', SELECTED_CONFIGS, '| MODE:', MODE)
print('Không MI/RFE/k-search. Không final fit hoặc Test tự động.')

## Cell 2 — Google Drive mount


In [ ]:
# Cell 2 — mount, không force_remount.
if globals().get('JOB9', {}).get('running') or globals().get('JOB', {}).get('running'):
    raise RuntimeError('Không mount khi worker đang chạy.')
from google.colab import drive
drive.mount('/content/drive')
if not DRIVE.is_dir():
    raise RuntimeError('MyDrive chưa truy cập được; dừng setup.')
print(DRIVE)

## Cell 3 — Repository và commit verification


In [ ]:
# Cell 3 — checkout Stage9 riêng, không sửa checkout A1/B đang có.
if JOB9.get('running'):
    raise RuntimeError('Worker đang chạy.')
if not re.fullmatch(r'[0-9a-f]{40}', APPROVED_COMMIT):
    raise ValueError('Điền SHA40 mới sau khi push Stage9 ở Cell 1.')
REPO = Path('/content/CICFLOW_STAGE9_' + APPROVED_COMMIT[:12])
def git_checked(repo, *args):
    return subprocess.run(['git', '-C', str(repo), *args], check=True, text=True, capture_output=True).stdout.strip()
def checkout_exact(repo, commit):
    if not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise ValueError('Saved commit không hợp lệ.')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--', 'https://github.com/kasum00/RFforNetwork.git', str(repo)], check=True)
        subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', commit], check=True)
    if git_checked(repo, 'rev-parse', 'HEAD') != commit or git_checked(repo, 'status', '--porcelain'):
        raise RuntimeError('Checkout sai SHA hoặc không sạch; không tự reset.')
checkout_exact(REPO, APPROVED_COMMIT)
SOURCE_ROOT = REPO / 'code_CICFLOW'
if not (SOURCE_ROOT / 'stage9_svm_final/pipeline.py').is_file():
    raise RuntimeError('Commit chưa chứa Stage9.')
for name, module in list(sys.modules.items()):
    if name == 'stage9_svm_final' or name.startswith('stage9_svm_final.') or name == 'mi_rfe_svm_colab' or name.startswith('mi_rfe_svm_colab.'):
        file = getattr(module, '__file__', None)
        if file and SOURCE_ROOT.resolve() not in Path(file).resolve().parents:
            raise RuntimeError('Kernel giữ source checkout khác. Dừng worker rồi dùng session sạch.')
sys.path.insert(0, str(SOURCE_ROOT))
print('STAGE9 SOURCE:', SOURCE_ROOT, '| COMMIT:', APPROVED_COMMIT)

## Cell 4 — Dependency profile


In [ ]:
# Cell 4 — chỉ kiểm tra dependencies. KHÔNG pip install hoặc restart tự động.
from stage9_svm_final.environment import environment
ENVIRONMENT = environment()
print(json.dumps(ENVIRONMENT, indent=2))
if ENVIRONMENT['errors']:
    raise RuntimeError('Sai dependency profile. Xem hướng dẫn cài thủ công, restart rồi chạy lại setup.')
from stage9_svm_final.common import confirmation, digest, paths, read
from stage9_svm_final.controller import launch
from stage9_svm_final.notebook_helpers import metadata, start_queue, monitor, safe_launch
from stage9_svm_final.selection import discover
TEST_EVIDENCE = {'status': 'NOT_RUN'}
SELECTION_SOURCES, VERIFIED_SELECTIONS, PREFLIGHT, FINAL_VERIFY, TEST_READINESS = {}, {}, {}, {}, {}
FINAL_REQUESTS, TEST_REQUESTS = [], []

## Cell 5 — Dataset metadata


In [ ]:
# Cell 5 — metadata nhỏ của dataset; không mở bất kỳ Test Parquet nào.
DATASET_METADATA = metadata(SOURCE_ROOT, DATASET_ROOT, CONFIGS)
print(json.dumps(DATASET_METADATA, indent=2))

## Cell 6 — Discovery selection A1/B


In [ ]:
# Cell 6 — kiểm kê chỉ đọc. completion_present không phải verify PASS.
DISCOVERED = discover(OUTPUT_ROOT)
for row in DISCOVERED:
    print(json.dumps(row, ensure_ascii=False))
for config in SELECTED_CONFIGS:
    print('CHOSEN:', METHOD, config, SELECTION_RUN_IDS[METHOD][config])
print('Nếu tên run chưa đúng, sửa Cell 1. Không nhập best_k thủ công.')

## Cell 7 — Read-only selection verification


In [ ]:
# Cell 7 — checkout source gốc theo saved commit và gọi verifier A1/B riêng.
if JOB9.get('running'):
    raise RuntimeError('Worker đang chạy.')
for config in SELECTED_CONFIGS:
    matches = [r for r in DISCOVERED if r['method'] == METHOD and r['config'] == config and r['run_id'] == SELECTION_RUN_IDS[METHOD][config]]
    if len(matches) != 1 or not matches[0].get('completion_present'):
        raise RuntimeError(f'BLOCKED_SELECTION_INCOMPLETE: {METHOD}/{config}; không chạy lại selection.')
    commit = matches[0]['commit']
    repo = Path('/content/STAGE9_SELECTION_SOURCE_' + commit[:12])
    checkout_exact(repo, commit)
    SELECTION_SOURCES[config] = repo / 'code_CICFLOW'
def request_for(config, mode='preflight'):
    return {
        'source': str(SOURCE_ROOT), 'git_commit': APPROVED_COMMIT,
        'selection_source': str(SELECTION_SOURCES[config]),
        'dataset': str(DATASET_ROOT), 'output': str(OUTPUT_ROOT), 'local_root': str(LOCAL_ROOT),
        'method': METHOD, 'config': config, 'run_id': FINAL_RUN_IDS[METHOD][config],
        'selection_run_id': SELECTION_RUN_IDS[METHOD][config],
        'evidence_root': EVIDENCE_ROOTS.get((METHOD, config)),
        'production': True, 'mode': mode, 'budget': dict(BUDGET), 'heartbeat_seconds': 15,
        'unit_tests': TEST_EVIDENCE, 'enable_final_fit': False, 'enable_test_evaluation': False,
        'final_confirmation': '', 'test_confirmation': '',
        'previous_worker_stopped': PREVIOUS_WORKER_STOPPED.get((METHOD, config)),
    }
VERIFIED_SELECTIONS = {}
for config in SELECTED_CONFIGS:
    value = safe_launch(request_for(config, 'verify-selection'))
    VERIFIED_SELECTIONS[config] = value
    print(config, 'VERIFIED best_k:', value['identity']['best_k'], '| features:', value['identity']['features'])

## Cell 8 — Synthetic unit tests


In [ ]:
# Cell 8 — synthetic tests có log. Không production SVC.fit/dataset Test.
TEST_EVIDENCE = launch(request_for(SELECTED_CONFIGS[0]), tests=True)
print('NATIVE EXIT:', TEST_EVIDENCE.get('native_exit_code'), '| STATUS:', TEST_EVIDENCE.get('status'))
if TEST_EVIDENCE.get('native_exit_code') != 0 or TEST_EVIDENCE.get('status') != 'PASS':
    raise RuntimeError('Unit tests chưa PASS; gửi toàn bộ log, không bỏ gate.')

## Cell 9 — Final-fit preflight


In [ ]:
# Cell 9 — preflight cho final fit, chưa training.
PREFLIGHT = {}
for config in SELECTED_CONFIGS:
    request = request_for(config)
    request['preflight_for'] = 'final-fit'
    PREFLIGHT[config] = launch(request)
print('Preflight xong; xem READY/BLOCKED tại Cell 10.')

## Cell 10 — READY / BLOCKED


In [ ]:
# Cell 10 — READY/BLOCKED. Exit 0 không tự chứng minh khoa học PASS.
for config in SELECTED_CONFIGS:
    value = PREFLIGHT[config]
    ready = value.get('native_exit_code') == 0 and value.get('status') == 'PASS'
    print(METHOD, config, 'READY' if ready else 'BLOCKED', value.get('reason', ''))
    if ready:
        print('  best_k:', value['identity']['best_k'], '| selection checkpoint:', value['identity']['selection_checkpoint_sha256'])
if not all(PREFLIGHT[c].get('native_exit_code') == 0 and PREFLIGHT[c].get('status') == 'PASS' for c in SELECTED_CONFIGS):
    raise RuntimeError('Có BLOCKED. Không đổi identity hoặc tự chạy lại selection.')

## Cell 11 — Xác nhận final fit


In [ ]:
# Cell 11 — chủ động cho phép FINAL FIT của METHOD đang chọn.
ENABLE_FINAL_FIT = False  # đổi True khi đã xem Cell 10.
FINAL_ACTION = 'final-fit'  # 'resume-final' khi tiếp tục final fit, không thay Run ID.
FINAL_REQUESTS = []
if ENABLE_FINAL_FIT:
    if JOB9.get('running') or FINAL_ACTION not in ('final-fit', 'resume-final'):
        raise RuntimeError('Đang có worker hoặc sai action.')
    for config in SELECTED_CONFIGS:
        if PREFLIGHT[config].get('status') != 'PASS' or PREFLIGHT[config].get('native_exit_code') != 0:
            raise RuntimeError('Preflight chưa READY.')
        request = request_for(config, FINAL_ACTION)
        expected = confirmation(request, 'FINAL_FIT')
        print('Nhập đúng:', expected)
        typed = input().strip()
        if typed != expected:
            raise RuntimeError('Câu xác nhận không khớp; chưa final fit.')
        request.update(enable_final_fit=True, final_confirmation=typed)
        FINAL_REQUESTS.append(request)
else:
    print('FINAL FIT DISABLED; không training.')

## Cell 12 — Final fit và heartbeat


In [ ]:
# Cell 12 — final fit tuần tự, heartbeat đo thực tế. Không Test tự động.
if FINAL_REQUESTS:
    if JOB9.get('running'):
        raise RuntimeError('Worker còn chạy; không mở queue thứ hai.')
    JOB9 = start_queue(FINAL_REQUESTS)
    monitor(JOB9)
else:
    print('Chưa có final requests được xác nhận ở Cell 11.')
# Nếu chỉ dừng monitor: chạy monitor(JOB9) lại. Không restart queue khi JOB9['running']=True.

## Cell 13 — Verify final model


In [ ]:
# Cell 13 — verify final model/scaler/checkpoint, không đọc Test.
if JOB9.get('running'):
    raise RuntimeError('Chờ final worker kết thúc.')
FINAL_VERIFY = {}
for config in SELECTED_CONFIGS:
    FINAL_VERIFY[config] = safe_launch(request_for(config, 'verify-final'))
    print(METHOD, config, FINAL_VERIFY[config]['phase_status'])

## Cell 14 — Test readiness audit


In [ ]:
# Cell 14 — audit Test ledger riêng từng pair. Chưa mở Test.
TEST_READINESS = {}
for config in SELECTED_CONFIGS:
    TEST_READINESS[config] = launch(request_for(config, 'test-readiness'))
    print(METHOD, config, TEST_READINESS[config].get('test_state', TEST_READINESS[config].get('phase_status')), TEST_READINESS[config].get('status'))
print('Claim không rõ/interrupted: BLOCKED, không retry. Completed: chỉ verify kết quả.')

## Cell 15 — Xác nhận Test riêng A1/B


In [ ]:
# Cell 15 — xác nhận TEST RIÊNG cho A1 hoặc B. Không chạy ngay sau final fit.
ENABLE_TEST_EVALUATION = False  # chủ động đổi True sau Cell 13–14.
TEST_CONFIRMATION = ''
TEST_REQUESTS = []
if ENABLE_TEST_EVALUATION:
    if JOB9.get('running'):
        raise RuntimeError('Worker còn chạy.')
    for config in SELECTED_CONFIGS:
        ready = TEST_READINESS[config]
        if ready.get('status') != 'PASS' or ready.get('native_exit_code') != 0:
            raise RuntimeError('Test readiness bị BLOCKED.')
        if ready.get('phase_status') == 'TEST_COMPLETE':
            print(METHOD, config, 'Đã có Test verified: dùng Cell 17, không evaluation lại.')
            continue
        request = request_for(config, 'test')
        expected = confirmation(request, 'TEST_ONCE')
        print('Nhập đúng:', expected)
        TEST_CONFIRMATION = input().strip()
        if TEST_CONFIRMATION != expected:
            raise RuntimeError('Không khớp xác nhận Test; chưa mở Test.')
        request.update(enable_test_evaluation=True, test_confirmation=TEST_CONFIRMATION)
        TEST_REQUESTS.append(request)
else:
    print('TEST EVALUATION DISABLED.')

## Cell 16 — Test evaluation một lần


In [ ]:
# Cell 16 — Test tối đa một lần cho từng pair đã xác nhận.
if TEST_REQUESTS:
    if JOB9.get('running'):
        raise RuntimeError('Không mở thêm worker.')
    JOB9 = start_queue(TEST_REQUESTS)
    monitor(JOB9)
else:
    print('Không có Test requests được xác nhận. Không Test tự động.')

## Cell 17 — Verify saved Test artifacts


In [ ]:
# Cell 17 — verify saved prediction/metrics/checkpoints, không predict hoặc mở Test dataset lại.
if JOB9.get('running'):
    raise RuntimeError('Worker chưa kết thúc.')
TEST_VERIFIED = {}
for config in SELECTED_CONFIGS:
    TEST_VERIFIED[config] = launch(request_for(config, 'verify-test'))
    print(METHOD, config, TEST_VERIFIED[config].get('status'), TEST_VERIFIED[config].get('phase_status'))

## Cell 18 — Bảng tám kết quả độc lập


In [ ]:
# Cell 18 — tổng hợp đủ tám dòng, thiếu Test verified thì để trống.
if JOB9.get('running'):
    raise RuntimeError('Chờ worker trước khi tổng hợp.')
from stage9_svm_final.reports import collect, export
from IPython.display import display
import pandas as pd
ROWS, REPORT_EVIDENCE = collect(OUTPUT_ROOT)
display(pd.DataFrame(ROWS))
print('Chỉ VERIFIED_TEST là Test metrics đã xác minh. Không lấy Validation thay thế.')

## Cell 19 — Export CSV / JSON / Markdown


In [ ]:
# Cell 19 — export mới, không ghi đè lịch sử. Không training/evaluation.
if JOB9.get('running'):
    raise RuntimeError('Worker còn chạy.')
EXPORTED = export(OUTPUT_ROOT)
print('REPORT DIRECTORY:', EXPORTED['directory'])
print('A1:', str(Path(EXPORTED['directory']) / 'SVM_A1_TEST_RESULTS.csv'))
print('B :', str(Path(EXPORTED['directory']) / 'SVM_B_TEST_RESULTS.csv'))
print('COMPARE:', str(Path(EXPORTED['directory']) / 'SVM_A1_B_COMPARISON.csv'))
print('AGGREGATE LOG:', DRIVE / 'code_CICFLOW/logs/ALL_EXECUTION_RESULTS.log')
print('Để làm phương án còn lại: chờ worker dừng, đổi METHOD ở Cell 1 và chạy setup lại; không trộn artifacts.')